In [ ]:
import os, pathlib, tarfile, shutil
base='/kaggle/input/datasets/asoksci/eeg-dementia-coursach-full'
prev='/kaggle/input/eeg-dementia-lead3'
out='/kaggle/working/repo'
os.makedirs(out, exist_ok=True)
import pathlib as pl
for tar in pl.Path(base).rglob('*.tar'):
    try:
        tf=tarfile.open(tar); tf.extractall(out); tf.close()
    except: pass
for p in ['configs','src','reports','figures']:
    s=pl.Path(base)/p
    if s.exists(): shutil.copytree(s, pl.Path(out)/p, dirs_exist_ok=True)
os.makedirs(out+'/src', exist_ok=True)
for f in pl.Path(base).rglob('src/*.py'):
    shutil.copy(f, pl.Path(out)/'src'/f.name)
import os as _os; _os.chdir(out)
!mkdir -p Processed/L400 Processed/ADFTD dataset/ADFTD
!cp -r /kaggle/input/eeg-dementia-lead3/repo/Processed/L400/ADFTD/* Processed/L400/ADFTD/ 2>&1 | head -5
!cp -r /kaggle/input/eeg-dementia-lead3/repo/Processed/L400/ADFTD/* Processed/ADFTD/ 2>&1 | head -5
!ls -lh Processed/ADFTD/meta.json Processed/L400/ADFTD/meta.json 2>&1 | head -10
!cat Processed/ADFTD/meta.json | head -n 20
!cat Processed/L400/ADFTD/meta.json | head -n 20
!pip install -q huggingface_hub einops -q
!git clone https://github.com/DL4mHealth/LEAD.git 2>&1 | tail -3
!mkdir -p checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41
!python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='YiheWang/LEAD', filename='checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41/checkpoint.pth', local_dir='.', local_dir_use_symlinks=False)" 2>&1 | tail -5
!ls -lh checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41/checkpoint.pth 2>&1 | head
!python LEAD/run.py --method LEAD --task_name finetune --is_training 1 --root_path ./Processed --data_path L400 --model_id P-11-b1024-p50-g16-learnable-3D-F-ADFTD-Multi --model LEAD --data MultiDatasets --training_dataset ADFTD --checkpoints_path ./checkpoints/LEAD/pretrain/LEAD/P-11-b1024-p50-g16-learnable-3D/nh8_el12_dm128_df256_seed41/checkpoint.pth --e_layers 12 --batch_size 64 --n_heads 8 --d_model 128 --d_ff 256 --patch_len 50 --stride 50 --temporal_pos_type learnable --channel_pos_type 3D --use_sampling_embedding --sampling_rate_list 200,100,50 --group_shuffle --group_size 4 --use_subject_loss --use_subject_vote --classify_choice multi_class --train_epochs 2 --patience 5 --cross_val mccv --itr 1 --learning_rate 0.0001 --seq_len 400 --enc_in 19 2>&1 | tee lead_mccv.log
!tail -n 300 lead_mccv.log
print('finetune v7 done')
